# 开发期现金与SOXL选择联合弱日的事前状态匹配

只用2018–2020逐层点时Qlib账户和事前已知字段。以年度、前SOXX20日趋势符号、开盘后风险状态分层，列出未来双负弱日及正超额强日的支持和预选坐标差；不训练、不从2021/2022挑特征。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import json,hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/"research/stage1_v5_oof_joint_state_match_20261030"
plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
for role in ("component_source","frame_source"):
 spec=plan[role];source=root/spec["card"]
 assert sha(source/"summary.json")==spec["summary_sha256"] and sha(source/"verification_manifest.json")==spec["manifest_sha256"]
 assert json.loads((source/"nas_binding.json").read_text())["evidence"]["sha256"]==spec["nas_archive_sha256"]
assert sha(root/plan["provider_uri"]/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
a=plan["component_source"];b=plan["frame_source"]
components,ar=load_verified_recorder_objects(root/a["card"],expected_recorder_id=a["recorder_id"],artifacts=["analysis/daily"],expected_manifest_sha256=a["manifest_sha256"])
frames,br=load_verified_recorder_objects(root/b["card"],expected_recorder_id=b["recorder_id"],artifacts=["inputs/covered_frame"],expected_manifest_sha256=b["manifest_sha256"])
daily=components["analysis/daily"].reset_index(drop=True);frame=frames["inputs/covered_frame"].reset_index(drop=True)
assert len(daily)==len(frame)==1056 and daily.trade_date.equals(frame.trade_date)
assert pd.to_datetime(frame.feature_available_at).lt(pd.to_datetime(frame.decision_at)).all()
mask=daily.trade_date.dt.year.isin(plan["discovery_years"])
daily=daily.loc[mask].copy().reset_index(drop=True);frame=frame.loc[mask].copy().reset_index(drop=True)
assert daily.trade_date.dt.year.isin(plan["discovery_years"]).all()
assert set(plan["preselected_coordinates"]).issubset(set(frame.columns)|set(daily.columns))
x=pd.DataFrame({"trade_date":daily.trade_date,"calendar_year":daily.trade_date.dt.year,
    "cash_log":daily.cash_log,"soxl_selection_log":daily.soxl_selection_log,"paired_log_excess":daily.paired_log_excess,
    "predecision_risk_state_nonzero":daily.risk_state.ne(0),"prior_soxx_ret20_sign":np.where(frame.soxx_ret20.gt(0),"positive","nonpositive")})
for name in plan["preselected_coordinates"]:
 x[name]=daily[name].to_numpy(float) if name in daily.columns else frame[name].to_numpy(float)
x["weak"]=x.cash_log.lt(0)&x.soxl_selection_log.lt(0)
x["strong"]=x.paired_log_excess.gt(0)&~x.weak
# Native Qlib DatasetH keeps source features, future-labelled diagnostics and dates aligned.
packed=pd.concat({"feature":x[plan["preselected_coordinates"]],"label":x[["weak","strong"]],"metadata":x[["trade_date","calendar_year","prior_soxx_ret20_sign","predecision_risk_state_nonzero"]]},axis=1)
packed.index=pd.MultiIndex.from_arrays([x.trade_date,["STAGE1"]*len(x)],names=["datetime","instrument"])
dataset=DatasetH(handler=DataHandlerLP.from_df(packed),segments={"discovery":("2018-10-15","2020-12-31")})
prepared=dataset.prepare("discovery",col_set=["feature","label","metadata"])
assert len(prepared)==len(x)
peryear=[]
for year,g in x.groupby("calendar_year",sort=True):
 peryear.append({"year":int(year),"days":len(g),"weak_days":int(g.weak.sum()),"strong_days":int(g.strong.sum()),
    "both_negative_cash_selection_days":int((g.cash_log.lt(0)&g.soxl_selection_log.lt(0)).sum())})
cell=[];differences=[]
for key,g in x.groupby(plan["match_cells"],sort=True,observed=True):
 weak=g.loc[g.weak];strong=g.loc[g.strong]
 row={"calendar_year":int(key[0]),"prior_soxx_ret20_sign":str(key[1]),"predecision_risk_state_nonzero":bool(key[2]),
      "days":len(g),"weak_days":len(weak),"strong_days":len(strong),
      "matched":len(weak)>=plan["minimum_each_class_per_cell"] and len(strong)>=plan["minimum_each_class_per_cell"]}
 cell.append(row)
 if row["matched"]:
  for name in plan["preselected_coordinates"]:
   differences.append({**{k:row[k] for k in ("calendar_year","prior_soxx_ret20_sign","predecision_risk_state_nonzero")},
       "feature":name,"weak_mean":float(weak[name].mean()),"strong_mean":float(strong[name].mean()),
       "weak_minus_strong":float(weak[name].mean()-strong[name].mean()),"weight":min(len(weak),len(strong))})
cells=pd.DataFrame(cell);diff=pd.DataFrame(differences)
summary=[]
if not diff.empty:
 for name,g in diff.groupby("feature",sort=True):
  summary.append({"feature":name,"matched_cells":len(g),"weighted_weak_minus_strong":float(np.average(g.weak_minus_strong,weights=g.weight)),
    "positive_cells":int(g.weak_minus_strong.gt(0).sum()),"negative_cells":int(g.weak_minus_strong.lt(0).sum()),
    "discovery_std":float(x[name].std()),"standardized_difference":float(np.average(g.weak_minus_strong,weights=g.weight)/x[name].std()) if x[name].std()>0 else None})
summary=pd.DataFrame(summary,columns=["feature","matched_cells","weighted_weak_minus_strong","positive_cells","negative_cells","discovery_std","standardized_difference"])
cells.to_csv(card/"matched_cells.csv",index=False)
summary.to_csv(card/"coordinate_summary.csv",index=False)
audit={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"decision":"descriptive_matched_state_support_no_feature_admission" if not diff.empty else "matched_state_support_failed_no_feature_admission",
 "source_readback_entries":{"components":len(ar),"frame":len(br)},"discovery_years":plan["discovery_years"],
 "reserved_years_not_evaluated":plan["reserved_years"],"year_counts":peryear,
 "matched_cells":int(cells.matched.sum()),"matched_weak_days_with_duplicates":int(cells.loc[cells.matched,"weak_days"].sum()),
 "matched_strong_days_with_duplicates":int(cells.loc[cells.matched,"strong_days"].sum()),
 "coordinate_summary":summary.to_dict("records"),"new_model_fits":0,"new_native_accounts":0,"candidate_feature_selected":False,
 "limits":["Outcome-defined weak/strong groups and matching are descriptive, not a tradable switch.","Daily signs are short-horizon physical account components; 931 rolling windows overlap.","2018 partial sample and 2020 recovery residual require separate reporting; 2021/2022 outcomes not inspected."]}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({"decision":audit["decision"],"year_counts":peryear,"matched_cells":audit["matched_cells"],"coordinates":summary.to_dict("records")[:5]},ensure_ascii=False))

## 中文结论

2018–2020逐层点时558日中，现金机会与SOXL选择同日双负仅11日（2018零、2019五、2020六）；同年、事前SOXX20日收益符号及开盘后风险状态分格，没有格子达到弱/强各10日的预定支持门。2019五个均在正趋势且风险状态零格，2020六个分布在另两类格，不能形成稳定开关。先前94个双负滚动窗口高度重叠，不是94个独立双负日。决定matched_state_support_failed_no_feature_admission；零拟合/账户，2021/2022未使用，不从本卡挑因子。下一步用现金与SOXL两项连续数值价值分别建模并在同一账户目标联合检验，但须先过旧路线去重与源时钟门。开发调整价/分红资格和SOXX0/2/5bp仍未过。